# 01. Auto Loader でファイルを取り込む

Auto Loader は、ストレージに置かれたファイルを**増分で**取り込む仕組みです。
一度読んだファイルを覚えていて、次に実行したときは新しく増えたファイルだけを読みます。

このノートブックで確かめること:

1. JSONファイルをDeltaテーブルに取り込む
2. 同じコードをもう一度実行すると何が起きるか
3. 「どこまで読んだか」をどうやって覚えているのか

**前提**: `00_setup` を実行して、カタログとVolumeを作ってあること。

## 準備

このノートブックはローカルのPythonで動きますが、Sparkの処理自体はDatabricks側で実行されます。
その橋渡しをするのが databricks-connect で、`DatabricksSession` がその入口です。

なお、VS CodeのDatabricks拡張機能を使っていると `spark` は自動で用意されるため、
本当はこのセルを書かなくても動きます。ここでは何が起きているかを見せるために明示的に作ります。

In [1]:
from databricks.connect import DatabricksSession

# serverless(True) = Databricks側のサーバーレスコンピュートに接続する
spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

In [2]:
# ファイル操作をDatabricks側に対して行うための道具
# ローカルから /Volumes/... を普通の open() で読み書きすることはできないため、これを経由する
from databricks.sdk.errors import NotFound
from databricks.sdk.runtime import dbutils

CATALOG = "tech_survey"

# 取り込み先のテーブル
TABLE = f"{CATALOG}.bronze.orders_raw"

# 取り込み元のフォルダ - 他のノートブックとファイルが混ざらないよう、トピック名で分ける
LANDING = f"/Volumes/{CATALOG}/ops/landing/01_auto_loader"

# チェックポイント = Auto Loaderが「どこまで読んだか」を記録する場所 (後で詳しく見ます)
CHECKPOINT = f"/Volumes/{CATALOG}/ops/checkpoints/01_auto_loader"

## 1. 取り込み元のファイルを用意する

まず前回の実行結果を消します。こうしておくと、何度やり直しても同じ状態から始められます。
消すのはこのノートブック専用のフォルダとテーブルだけなので、他のトピックには影響しません。

In [3]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

# 第2引数の True = フォルダの中身ごと消す
# rm は対象が無いとエラーになる。初回実行ではまだフォルダが無いので、その場合は無視する
for path in (LANDING, CHECKPOINT):
    try:
        dbutils.fs.rm(path, True)
    except NotFound:
        pass

True

In [4]:
import json
import random
import uuid

# 注文データを50件作る
rows = []
for _ in range(50):
    rows.append(
        {
            "order_id": str(uuid.uuid4()),
            "product": random.choice(["laptop", "monitor", "keyboard"]),
            "amount": random.randint(1000, 50000),
        }
    )

# JSON Lines形式 = 1行に1レコード。Auto Loaderが読むのはこの形式
text = "\n".join(json.dumps(row) for row in rows)

dbutils.fs.put(f"{LANDING}/orders_1.json", text, True)

True

In [5]:
# 今フォルダに何があるか確認する
dbutils.fs.ls(LANDING)

[FileInfo(path='/Volumes/tech_survey/ops/landing/01_auto_loader/orders_1.json', name='orders_1.json', size=4597, modificationTime=1789200890000)]

## 2. Auto Loader で読む

`spark.read` ではなく **`spark.readStream`** を使います。この2つの違いが Auto Loader の肝です。

- `spark.read` … 毎回フォルダ全体を読む。前回何を読んだかは覚えていない
- `spark.readStream` … 前回の続きから読む。そのために「どこまで読んだか」を記録する

形式に `cloudFiles` を指定すると、この readStream が Auto Loader として動きます。

オプションの意味:

- `cloudFiles.format` … 読むファイルの形式。今回はJSON
- `cloudFiles.schemaLocation` … 推論した列名・型を保存しておく場所。
  JSONには型の情報がないので、Auto Loaderは中身を見て型を推測します。
  その結果をここに保存し、次回は読み直さずに再利用します

In [ ]:
# この時点ではまだ読み込みは始まりません。「こう読む」という定義を作っているだけです。
df = (
    spark.readStream.format("cloudFiles")  # `cloudFiles`で、Auto Loaderを使うことを宣言する
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/_schema")
    .load(LANDING)
)

## 3. テーブルに書き出す

読み取りの定義に、書き出し先を繋いで初めて処理が動きます。

**チェックポイント** がここで効いてきます。Auto Loader は処理したファイル名をチェックポイントに
記録します。次に同じチェックポイントを指定して実行すると、記録済みのファイルは飛ばして、
新しいファイルだけを読みます。これが「増分で取り込む」の中身です。
逆に言うと、チェックポイントを消せば最初から読み直しになります。

**トリガー** は処理をいつ動かすかの指定です。`availableNow=True` は
「今ある未処理のファイルを全部処理したら終了する」という意味で、
ノートブックでの確認や1日1回のバッチ処理に向いています。
指定しないと処理が終わらず、新しいファイルを待ち続けます。

### チェックポイントの中身を確認する

中に入っているもの（4つ）

- `metadata`：このストリームの識別子
- `offsets/`：バッチごとに「ソースのどこまでを処理対象にしたか」を記録。処理の前に書かれる
- `commits/`：バッチごとに「書き込みまで完了した」記録。処理の後に書かれる
- `sources/0/`：ソース固有の状態。Auto Loaderの場合、ここに「見たファイル一覧」が入る

In [ ]:
# 以下の読み書きの定義をした時点で、処理は走り出します。終わるまで待つ必要があります。
query = (
    df.writeStream.option("checkpointLocation", CHECKPOINT)  # チェックポイントの場所を指定する
    .trigger(availableNow=True)
    .toTable(TABLE)
)

query.awaitTermination()  # .awaitTermination()はqueryの処理が終わるまでここで待つ

In [9]:
display(spark.table(TABLE))

,amount,order_id,product,_rescued_data
0,28032,1326cbed-d357-45c4-8c76-e35e8eb90608,monitor,None
1,44282,d60b4a8c-c183-45a9-bdb9-5c973436db5b,laptop,None
2,16273,867c7690-9720-465a-b04f-e994f3f40b89,monitor,None
3,28424,1ed2809d-fa0a-4cb9-922a-3adeed20d605,laptop,None
4,27429,8d35e979-bd6e-4112-abcb-621d91a2d648,keyboard,None
5,18339,448805e1-4c31-4a53-b585-476307b97ce9,monitor,None
6,15346,564647fe-c0e1-4db7-8265-83af1d28fc17,keyboard,None
7,17800,9907f01b-106e-4751-801e-c84e8eb530e2,keyboard,None
8,49224,e6263e33-01d1-47a6-8dc3-c4032ae466c4,monitor,None
9,26592,9e227f8e-da32-4012-83f1-affe6335c330,laptop,None


In [10]:
spark.table(TABLE).count()

50

## 4. もう一度実行すると、どうなるか

ここが一番確かめたいところです。新しいファイルを1つ追加してから、
**さっきとまったく同じ取り込み処理** をもう一度動かします。

動かす前に予想してみてください。テーブルの件数はどうなるでしょうか。

- 追加した分だけ増える
- 最初のファイルも読み直されて倍近くになる

どちらになるか、そしてそれはなぜか。

In [11]:
# 30件の新しいファイルを追加する
rows = []
for _ in range(30):
    rows.append(
        {
            "order_id": str(uuid.uuid4()),
            "product": random.choice(["laptop", "monitor", "keyboard"]),
            "amount": random.randint(1000, 50000),
        }
    )

dbutils.fs.put(f"{LANDING}/orders_2.json", "\n".join(json.dumps(r) for r in rows), True)
dbutils.fs.ls(LANDING)

[FileInfo(path='/Volumes/tech_survey/ops/landing/01_auto_loader/orders_1.json', name='orders_1.json', size=4597, modificationTime=1789200890000),
 FileInfo(path='/Volumes/tech_survey/ops/landing/01_auto_loader/orders_2.json', name='orders_2.json', size=2755, modificationTime=1789213275000)]

In [12]:
# さっきと同じ内容。チェックポイントも同じ場所を指している
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/_schema")
    .load(LANDING)
)

query = (
    df.writeStream.option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TABLE)
)
query.awaitTermination()

In [13]:
spark.table(TABLE).count()

80

## 5. チェックポイントの中を見る

「どこまで読んだか」が実際にどう保存されているかを覗いてみます。

In [14]:
dbutils.fs.ls(CHECKPOINT)

[FileInfo(path='/Volumes/tech_survey/ops/checkpoints/01_auto_loader/metadata', name='metadata', size=45, modificationTime=1789201368000),
 FileInfo(path='/Volumes/tech_survey/ops/checkpoints/01_auto_loader/_schema/', name='', size=None, modificationTime=None),
 FileInfo(path='/Volumes/tech_survey/ops/checkpoints/01_auto_loader/commits/', name='', size=None, modificationTime=None),
 FileInfo(path='/Volumes/tech_survey/ops/checkpoints/01_auto_loader/offsets/', name='', size=None, modificationTime=None),
 FileInfo(path='/Volumes/tech_survey/ops/checkpoints/01_auto_loader/sources/', name='', size=None, modificationTime=None)]

## 6. チェックポイントを消すとどうなるか

「どこまで読んだか」の記録だけを消して、同じ取り込みをもう一度動かします。
テーブルと取り込み元のファイルはそのまま残します。

動かす前に予想してみてください。件数はどうなるでしょうか。

In [16]:
# チェックポイントだけを消す。テーブルとlandingのファイルはそのまま残る
try:
    dbutils.fs.rm(CHECKPOINT, True)
except NotFound:
    pass

True

In [17]:
# 3章とまったく同じ処理をもう一度動かす
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/_schema")
    .load(LANDING)
)

query = (
    df.writeStream.option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TABLE)
)
query.awaitTermination()

In [18]:
spark.table(TABLE).count()

160

結果を見てから読んでください。

チェックポイントには「どのファイルを処理済みか」が入っていました。それを消したので、
Auto Loaderから見ると全ファイルが初めて見るファイルになります。

ここで押さえておきたいのは、**Auto Loaderは書き込み先テーブルの中身を見ていない**ことです。
「このデータはもう入っているか」を確認してから書く、という動きはしません。
チェックポイントだけを頼りに、読むか読まないかを決めています。

つまり「同じ処理を何度動かしても結果が変わらない」ことを支えているのはチェックポイントであって、
テーブル側の重複チェックではありません。チェックポイントを消したり別の場所を指したりすると、
簡単に重複します。この話は `06_idempotent_writes` で詳しく扱います。

## 考えてみる

- `4.` の結果は予想どおりでしたか。違ったなら、どこが思い込みだったでしょうか
- `6.` で重複が起きたとして、それを防ぐにはどうすればよいでしょうか
- 1日1回だけ動かすバッチ処理を作るとしたら、この仕組みの何が嬉しいでしょうか

### 答え

**Q1. `4.` の結果は予想どおりだったか**

追加したファイルの分だけが増えます。最初のファイルは読み直されません。
チェックポイントに「このファイルは処理済み」と記録されているためです。

よくある思い込みは「フォルダ全体をもう一度読むはず」というものです。
普通の `spark.read` ならそうなりますが、`readStream` は前回の続きから読みます。

**Q2. `6.` の重複を防ぐには**

一番の基本は、チェックポイントを消さないこと、そして別のストリームと使い回さないことです。

やり直したい場合は、**チェックポイントとテーブルを必ずセットで作り直します**。
片方だけ消すと、今回のように記録と実データがずれて重複します。
`0.` のリセットセルで両方まとめて消しているのは、このためです。

書き込み側で防ぐ方法もあります。追記ではなく `MERGE` でキーを見て upsert すれば、
同じデータが2回来ても1件になります。この方法は `06_idempotent_writes` で扱います。

**Q3. 日次バッチで何が嬉しいか**

- 「前回どこまで処理したか」を自分で管理しなくてよい。日付でファイル名を絞る、といった処理が要らない
- ファイルが増えても差分しか読まないので、処理時間が増え続けない
- 失敗しても、次の実行が続きから拾ってくれる

## 後片付け

作ったものを消したいときに実行します。

In [ ]:
# このノートブックで作ったものを消す

# spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
# dbutils.fs.rm(LANDING, True)
# dbutils.fs.rm(CHECKPOINT, True)